# NLP Masterclass 01: Text Preprocessing & Tokenization

**Track:** Natural Language Processing (0 to Masterclass)  
**Prerequisites:** Python fundamentals  
**Difficulty:** ⭐ Beginner → Intermediate  
**Estimated Time:** 60–90 minutes

---

## 🎓 Socratic Deep Check

> *"GPT-4 doesn't read words — it reads **tokens**. The word 'unhappiness' might become ['un', 'happiness'] or ['un', 'happi', 'ness']. Why would a model intentionally break words into subwords instead of using whole words?"*

---

## Why This Matters for AI Engineers

Every LLM interaction starts with tokenization. API costs are per-token. Context windows are measured in tokens. Prompt engineering (NB24) requires understanding token boundaries. RAG chunking (NB28) depends on token counts.

**This notebook is the foundation for the entire LLMOps track (NB24-34).**

## Table of Contents
1. Text Cleaning & Normalization
2. Word-Level Tokenization & Its Limits
3. Subword Tokenization: BPE
4. Modern Tokenizers: HuggingFace's tokenizers library
5. Special Tokens & Chat Templates

---

### 🎓 Junior to Senior: Concept Bridge

**The Senior Perspective:** Juniors think of tokenization as `text.split(' ')`. Seniors know tokenization is a learned compression algorithm that controls vocabulary size, handles unseen words, determines API costs, and directly limits what a model can express.

**Mental Model:** Tokenization is like choosing an alphabet. Whole-word tokenization means memorizing every word in every language. Subword tokenization is like learning roots and prefixes — "unhappiness" = "un" + "happi" + "ness" — you can construct ANY word from pieces.

**Common Junior Pitfall:** Assuming 1 word = 1 token. In GPT-4, "Hello" is 1 token but "counterrevolutionary" is 4. This breaks cost estimates and context window calculations.

---

In [ ]:
!pip install -q transformers tiktoken regex

## 1 · Text Cleaning & Normalization

Real-world text is messy. Before any NLP, clean it.

| Problem | Example | Fix |
|---------|---------|-----|
| Mixed case | "Apple" vs "apple" | Lowercasing |
| Unicode | "café" vs "cafe\u0301" | Unicode normalization (NFKD) |
| HTML artifacts | "&amp;" "&lt;br&gt;" | Strip HTML |
| Extra whitespace | "hello   world" | Collapse |
| Emojis | "Great job! 🎉" | Keep or remove (domain-dependent) |

In [ ]:
import re
import unicodedata

def clean_text(text):
    """Production-grade text cleaning pipeline."""
    # 1. Unicode normalization (handles accented characters consistently)
    text = unicodedata.normalize('NFKD', text)
    
    # 2. Remove HTML tags
    text = re.sub(r'<[^>]+>', '', text)
    
    # 3. Remove HTML entities
    text = re.sub(r'&[a-zA-Z]+;', ' ', text)
    
    # 4. Collapse whitespace
    text = re.sub(r'\s+', ' ', text).strip()
    
    return text

# Test
messy = '  <p>The café &amp; résumé   had    \n\n problems</p>  '
clean = clean_text(messy)
print(f'Before: "{messy}"')
print(f'After:  "{clean}"')

# IMPORTANT: Modern LLMs (GPT-4, Llama) handle messy text fine.
# Cleaning is MORE important for: traditional ML, search indices, deduplication.
print('\n⚠️ For LLM prompts: minimal cleaning. For ML pipelines: aggressive cleaning.')

---
## 2 · Tokenization Approaches

### The Vocabulary Size Tradeoff

| Approach | Vocab Size | Handles Unseen Words? | Token/Word Ratio |
|----------|-----------|----------------------|------------------|
| Character-level | ~256 | ✅ Yes | ~5:1 (long sequences) |
| Word-level | ~500K+ | ❌ No (OOV problem) | ~1:1 |
| **Subword (BPE)** | **32K-100K** | **✅ Yes** | **~1.3:1** |

### BPE: Byte-Pair Encoding (Used by GPT, Llama)

**Algorithm:**
1. Start with individual characters as the vocabulary
2. Count the most frequent adjacent pair (e.g., "t" + "h" = "th")
3. Merge that pair into a single token
4. Repeat N times (N = desired vocab size - initial chars)

In [ ]:
# BPE from scratch — simplified version
from collections import Counter

def learn_bpe(corpus, num_merges=10):
    """Learn BPE merge rules from a corpus."""
    # Start: each word is split into characters + end-of-word marker
    words = {}
    for word in corpus.split():
        chars = tuple(list(word) + ['</w>'])
        words[chars] = words.get(chars, 0) + 1
    
    merges = []
    for i in range(num_merges):
        # Count all adjacent pairs
        pairs = Counter()
        for word, freq in words.items():
            for j in range(len(word) - 1):
                pairs[(word[j], word[j+1])] += freq
        
        if not pairs:
            break
        
        # Find most frequent pair
        best = pairs.most_common(1)[0]
        pair = best[0]
        merges.append(pair)
        
        # Merge that pair in all words
        new_words = {}
        merged = pair[0] + pair[1]
        for word, freq in words.items():
            new_word = []
            j = 0
            while j < len(word):
                if j < len(word)-1 and (word[j], word[j+1]) == pair:
                    new_word.append(merged)
                    j += 2
                else:
                    new_word.append(word[j])
                    j += 1
            new_words[tuple(new_word)] = freq
        words = new_words
        
        print(f'Merge {i+1}: "{pair[0]}" + "{pair[1]}" -> "{merged}"  (freq={best[1]})')
    
    return merges, words

corpus = "the cat sat on the mat the cat ate the rat"
print(f'Corpus: "{corpus}"\n')
merges, vocab = learn_bpe(corpus, num_merges=8)
print(f'\nFinal vocabulary: {set(tok for word in vocab for tok in word)}')

In [ ]:
# Real tokenizers: comparing models
import tiktoken
from transformers import AutoTokenizer

text = "The unbelievably counterintuitive transformer architecture revolutionized AI."

# GPT-4 tokenizer (tiktoken)
gpt4_enc = tiktoken.encoding_for_model('gpt-4')
gpt4_tokens = gpt4_enc.encode(text)
gpt4_decoded = [gpt4_enc.decode([t]) for t in gpt4_tokens]

print(f'Text: "{text}"')
print(f'\nGPT-4 ({len(gpt4_tokens)} tokens):  {gpt4_decoded}')

# Cost estimation (practical!)
price_per_1k = 0.03  # GPT-4 input pricing
prompt_tokens = len(gpt4_tokens)
print(f'\n💰 API Cost:')
print(f'  This sentence: {prompt_tokens} tokens = ${prompt_tokens * price_per_1k / 1000:.6f}')
print(f'  1000 similar sentences: ${1000 * prompt_tokens * price_per_1k / 1000:.4f}')
print(f'  1M calls/day: ${1e6 * prompt_tokens * price_per_1k / 1000:.0f}/day')

---
## 3 · Special Tokens & Chat Templates

LLMs use special tokens to structure conversations:

| Token | Purpose | Example |
|-------|---------|--------|
| `<bos>` | Beginning of sequence | Start of generation |
| `<eos>` | End of sequence | Stop generating |
| `<pad>` | Padding (make sequences equal length) | Batching |
| `[INST]` | Instruction marker (Llama chat) | User message start |
| `<|system|>` | System prompt (ChatML) | System message |

This directly connects to NB24 (Prompt Engineering) — understanding chat templates is essential for production prompts.

In [ ]:
# Chat template comparison
print('=== Chat Template Formats ===')
print()
print('--- Llama 3 (used in NB11 fine-tuning) ---')
print('<|begin_of_text|><|start_header_id|>system<|end_header_id|>')
print('You are a helpful assistant.<|eot_id|>')
print('<|start_header_id|>user<|end_header_id|>')
print('What is attention?<|eot_id|>')
print('<|start_header_id|>assistant<|end_header_id|>')
print()
print('--- ChatML (OpenAI) ---')
print('<|im_start|>system')
print('You are a helpful assistant.<|im_end|>')
print('<|im_start|>user')
print('What is attention?<|im_end|>')
print('<|im_start|>assistant')
print()
print('--- Connection to NB11 ---')
print('When LoRA fine-tuning in NB11, the SFTTrainer formats your data')
print('into the chat template automatically. Mismatched templates = garbage output.')

---
## ✅ Knowledge Check

### Q1: Why subwords?
Why does BPE tokenize "unhappiness" as ["un", "happi", "ness"] instead of one token?

<details><summary>👉 Answer</summary>

With word-level tokenization, you would need every form of every word in your vocabulary ("happy", "unhappy", "happiness", "unhappiness", "happier"...). Subword tokenization reuses morphological units: "un" + "happi" + "ness". This keeps vocabulary manageable (~32K tokens) while handling ANY word, including misspellings and new words.
</details>

### Q2: Token count surprise
"Hello" is 1 token, but " Hello" (with a space) is also 1 token (different ID). Why?

<details><summary>👉 Answer</summary>

BPE treats spaces as part of the token. "Hello" at the start of a sentence has no leading space (token ID 9906 in GPT-4). " Hello" in the middle of a sentence includes the space (token ID 22691). This is why prompt engineering spacing matters — spaces are encoded!
</details>

### Q3: Cost optimization
Your RAG system (NB28) sends 4000 tokens per query and handles 100K queries/day with GPT-4 ($0.03/1K tokens). What's the daily input cost? How would you reduce it?

<details><summary>👉 Answer</summary>

Daily cost: 100K × 4000 × $0.03/1000 = $12,000/day. Reduction strategies: (1) Use a cheaper model for simple queries (GPT-4-mini), (2) Reduce retrieved context in RAG, (3) Compress prompts (remove redundant instructions), (4) Cache frequent query results.
</details>

---

## 🔨 Practical Practice

### Exercise 1: Token Count Estimator
Write a function that takes a text and model name, returns the token count and estimated API cost. Test with GPT-4 and Llama 3 tokenizers.

### Exercise 2: BPE Analysis
Run the BPE algorithm on a paragraph of code (Python). How do code tokens differ from English tokens? What does this mean for code-generating LLMs?

### Exercise 3: Chat Template Converter
Write a function that converts a list of `{role, content}` messages into both Llama 3 and ChatML format strings.

---

**Next →** NLP 02: Word Embeddings & Word2Vec